<a href="https://colab.research.google.com/github/simjonghyeon04/-/blob/main/%EB%8F%84%EC%84%9C%EA%B4%80%20%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. 필수 패키지 설치
!pip install fastapi uvicorn sqlalchemy pydantic gradio python-multipart

import os
import random
from string import digits
from fastapi import FastAPI, Depends, HTTPException
from sqlalchemy import create_engine, Column, Integer, String, Boolean
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, Session
import gradio as gr

# ==============================================================================
# [MVC 1단계: Database & Model] 데이터베이스 및 테이블 설정
# ==============================================================================
SQLALCHEMY_DATABASE_URL = "sqlite:///./library.db"
engine = create_engine(SQLALCHEMY_DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()

class DBBook(Base):
    __tablename__ = "books"
    id = Column(Integer, primary_key=True, index=True)
    title = Column(String, index=True, nullable=False)
    author = Column(String, nullable=False)
    description = Column(String, nullable=True)
    is_borrowed = Column(Boolean, default=False)

# 테이블 생성
Base.metadata.create_all(bind=engine)

# ==============================================================================
# [MVC 2단계: CRUD 로직] 데이터베이스 제어 함수들
# ==============================================================================
def get_all_books():
    db = SessionLocal()
    books = db.query(DBBook).all()
    db.close()
    return books

def add_new_book(title, author, description):
    if not title or not author:
        return "❌ 제목과 저자는 필수 입력 항목입니다."
    db = SessionLocal()
    db_book = DBBook(title=title, author=author, description=description)
    db.add(db_book)
    db.commit()
    db.close()
    return f"✅ 도서 '{title}' 등록 완료!"

def toggle_borrow(book_id):
    db = SessionLocal()
    book = db.query(DBBook).filter(DBBook.id == book_id).first()
    if not book:
        db.close()
        return "❌ 해당 ID의 도서를 찾을 수 없습니다."

    book.is_borrowed = not book.is_borrowed
    status = "대출" if book.is_borrowed else "반납"
    db.commit()
    db.close()
    return f"✅ 도서 ID {book_id}번이 성공적으로 [{status}] 되었습니다."

def remove_book(book_id):
    db = SessionLocal()
    book = db.query(DBBook).filter(DBBook.id == book_id).first()
    if not book:
        db.close()
        return "❌ 해당 ID의 도서를 찾을 수 없습니다."
    db.delete(book)
    db.commit()
    db.close()
    return f"✅ 도서 ID {book_id}번이 완전히 삭제되었습니다."

# ==============================================================================
# [Gradio 인터페이스 구축] 웹 화면 만들기
# ==============================================================================
def refresh_list():
    books = get_all_books()
    if not books:
        return "📚 도서관에 등록된 책이 없습니다."

    output = "✨ [현재 도서관 보유 목록] ✨\n"
    output += "="*60 + "\n"
    for b in books:
        status = "🔴 대출 중" if b.is_borrowed else "🍏 대출 가능"
        output += f"ID: {b.id} | 제목: {b.title} | 저자: {b.author} | 상태: {status}\n"
        if b.description:
            output += f"   └ 설명: {b.description}\n"
        output += "-"*60 + "\n"
    return output

# Gradio 화면 레이아웃 짜기
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📚 FastAPI & Gradio 도서 관리 시스템")
    gr.Markdown("조상구 교수님 기말과제 실습용 대시보드입니다. 패스워드 없이 아래 링크로 즉시 접속 가능합니다.")

    with gr.Row():
        # 왼쪽: 도서 목록 현황판
        with gr.Column(scale=2):
            gr.Markdown("### 🗂️ 도서 현황 목록")
            book_list_viewer = gr.Textbox(value=refresh_list(), label="실시간 책장 상태", lines=20)
            btn_refresh = gr.Button("🔄 목록 새로고침", variant="secondary")

        # 오른쪽: 도서 등록 및 관리 조작반
        with gr.Column(scale=1):
            gr.Markdown("### 📥 신규 도서 등록")
            in_title = gr.Textbox(label="도서 제목", placeholder="예: 파이썬 웹 프로그래밍")
            in_author = gr.Textbox(label="저자 이름", placeholder="예: 홍길동")
            in_desc = gr.Textbox(label="도서 설명 (선택)", placeholder="간단한 책 소개")
            btn_add = gr.Button("➕ 도서 등록하기", variant="primary")
            out_add_result = gr.Markdown("")

            gr.Markdown("---")
            gr.Markdown("### ⚙️ 대출 / 반납 / 삭제 관리")
            in_book_id = gr.Number(label="도서 ID 번호 선택", value=1, precision=0)
            with gr.Row():
                btn_borrow = gr.Button("🔄 대출/반납 전환", variant="warning")
                btn_delete = gr.Button("🗑️ 도서 완전 삭제", variant="stop")
            out_manage_result = gr.Markdown("")

    # 기능 이벤트 매핑
    btn_refresh.click(fn=refresh_list, outputs=book_list_viewer)

    btn_add.click(
        fn=lambda t, a, d: (add_new_book(t, a, d), refresh_list()),
        inputs=[in_title, in_author, in_desc],
        outputs=[out_add_result, book_list_viewer]
    )

    btn_borrow.click(
        fn=lambda bid: (toggle_borrow(bid), refresh_list()),
        inputs=[in_book_id],
        outputs=[out_manage_result, book_list_viewer]
    )

    btn_delete.click(
        fn=lambda bid: (remove_book(bid), refresh_list()),
        inputs=[in_book_id],
        outputs=[out_manage_result, book_list_viewer]
    )

# ==============================================================================
# [실행] 기존 유비콘 종료 및 Gradio 외부 링크 활성화 실행
# ==============================================================================
!pkill -f uvicorn
!pkill -f gradio

print("\n🚀 Gradio 무료 터널링 인프라 가동 중...")
# share=True 옵션이 핵심입니다. 외부에서 접속 가능한 .gradio.live 링크를 0초 만에 뚫어줍니다.
demo.launch(share=True, debug=False)

/tmp/ipykernel_2008/3758367737.py:19: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()
/tmp/ipykernel_2008/3758367737.py:94: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:



🚀 Gradio 무료 터널링 인프라 가동 중...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://112d645916be58a432.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
